In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
from scipy.optimize import root_scalar

# Code finds minimum value based on the subharmonic frequency prediction
# for varying driving frequency


# Constants
g = 9.81
sigma = 0.072
rho = 1000
nu = 1e-6
#f_p = 20.0
#omega_p = 2 * np.pi * f_p
h_depth = 0.005

a_vals = np.linspace(0, 1000, 40)

def dispersion_relation(k):
    """Dispersion relation for gravity-capillary waves"""
    return np.sqrt(g * k + (sigma / rho) * k**3) * np.tanh(k * h_depth)

def inverse_dispersion_relation(omega, kmin=0, kmax=1e4):
    """Finds k from omega_d according to the dispersion relation"""
    sol = root_scalar(
        lambda k: dispersion_relation(k) - omega,
        bracket=[kmin, kmax],
        method='brentq')

    return sol.root

def get_gamma_exact(k):
    """Gamma factor according to a research paper"""
    return nu*k**2 * (2 + 1/(np.tanh(2*k*h_depth)*np.sinh(2*k*h_depth))) + np.sqrt(k*nu*np.sqrt(9.81*h_depth)/8)*2*k/np.sinh(2*k*h_depth)

def system_dynamics(y, t_scalar, k, A, omega_d):
    """Function guards the system dynamics and will be numerically solved"""
    h, v = y
    tanh_term = np.tanh(k * h_depth)
    omega0_sq = dispersion_relation(k)**2
    gamma = get_gamma_exact(k)

    # The forcing modulates gravity specifically
    # forcing = (A * k * tanh(kh)) * cos(omega_d * t)
    forcing = (A * k * tanh_term) * np.cos(omega_d * t_scalar)

    # Dynamics of system
    dhdt = v
    dvdt = -2 * gamma * v - (omega0_sq - forcing) * h
    return [dhdt, dvdt]

def instability_measure(k, A, omega_d):
    """Returns real part of eigenvalues"""
    T = 2 * np.pi / omega_d
    t = [0, T]
    # Monodromy matrix construction
    res1 = odeint(system_dynamics, [1.0, 0.0], t, args=(k, A, omega_d))[-1]
    res2 = odeint(system_dynamics, [0.0, 1.0], t, args=(k, A, omega_d))[-1]
    M = np.column_stack([res1, res2])
    eigenvalues = np.linalg.eigvals(M)
    return np.real(max(eigenvalues, key=lambda x: abs(x)))

def boundary_finder(k_val, omega_d):
    """Function that finds acceleration boundary for given k"""
    if isinstance(k_val, np.ndarray):
        k_val = k_val.item() # Convert [val] to val

    val_2 = instability_measure(k_val, a_vals[0], omega_d) # Start cycle properly
    a_2 = a_vals[0]
    for a in a_vals[1:]:
        val_1 = val_2 # Optimization to prevent calculating the same thing twice
        val_2 = instability_measure(k_val, a, omega_d)
        a_1 = a_2
        a_2 = a

        # Checks boundary of stability with instability
        if (abs(val_2)-1)*(abs(val_1)-1) < 0:
            sol = root_scalar(lambda a: abs(instability_measure(k_val, a, omega_d)) - 1,
                              bracket=[a_1, a_2], method='brentq')
            return sol.root

def minimum_finder(f_d: float):
    """Finds minimum acceleration for given driving frequency"""
    omega_d = 2*np.pi*f_d
    k_guess = inverse_dispersion_relation(omega_d/2)

    # Finds local minimum close to given k_guess
    res = minimize(boundary_finder, k_guess, args=(omega_d))
    k_opt = res.x[0]
    f_opt = dispersion_relation(k_opt)/(2*np.pi)
    a_min = res.fun
    return a_min, k_opt, f_opt

# Execute code on given linspace
f_d_vals = np.linspace(5, 50, 20)
minima = []
a = []
k = []
f = []

for f_d in f_d_vals:
    a_res, k_res, f_res = minimum_finder(f_d)
    a.append(a_res)
    k.append(k_res)
    f.append(f_res)

# Compare to analytic model
f_d_vals2 = np.linspace(5, 50, 50)
omega_resonant = np.pi*f_d_vals2
k = np.zeros(len(f_d_vals2))
for i in range(len(k)):
    k[i] = inverse_dispersion_relation(omega_resonant[i])
gamma = get_gamma_exact(k)
a_c = 4*gamma*omega_resonant/(g*k*np.tanh(k*h_depth))

# Plotting a_c versus f_d and formatting
plt.plot(f_d_vals2, a_c, color="red", label="Analytic approximation", linestyle="--")
plt.plot(f_d_vals, np.array(a)/g, label="Numerical approximation", linestyle="--")
plt.xlabel("Driving frequency $f_d$ (Hz)")
plt.ylabel("On-set Gamma $\Gamma_z$")
plt.grid(linestyle="--", alpha=0.5)
plt.legend()
plt.show()

# Plotting f_res versus f_d and formatting
plt.plot(f_d_vals, f, color="green", linestyle="--", label="Numerical model")
plt.plot(f_d_vals, 0.5*f_d_vals, color="red", linestyle="--", label="Subharmonic model ($f_{res} = 0.5 f_d$)")
plt.xlabel("Driving frequency $f_d$ (Hz)")
plt.ylabel("Optimal frequency $f_{res}$ (Hz)")
plt.grid(linestyle="--", alpha=0.5)
plt.legend()
plt.show()